# L12 – Lösningar
Kör först setup-cellen nedan. Den återskapar data, fönster och hjälpfunktioner från lektionen.

In [ ]:
import copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
CSV = "../../AI25-Deep_learning/Data/CTA_-_Ridership_-_Daily_Boarding_Totals_20260118.csv"
df = pd.read_csv(CSV, thousands=",", parse_dates=["service_date"], date_format="%m/%d/%Y")
series = df.sort_values("service_date").set_index("service_date").loc["2010":"2019", "total_rides"].asfreq("D") / 1e6
TRAIN_END, VAL_END = "2016-12-31", "2018-12-31"
mean, std = series[:TRAIN_END].mean(), series[:TRAIN_END].std()
z = ((series - mean) / std).to_numpy(dtype="float32")

def make_windows(values, window):
    X = np.stack([values[i - window:i] for i in range(window, len(values))])
    y = values[window:]
    return (X[..., None] if X.ndim == 2 else X), y

def split_masks(window):
    d = series.index[window:]
    return d <= TRAIN_END, (d > TRAIN_END) & (d <= VAL_END), d > VAL_END

def to_rides(v):
    return v * std + mean

def mae_rides(pred_z, true_z):
    return np.abs(to_rides(pred_z) - to_rides(true_z)).mean() * 1e6

class Forecaster(nn.Module):
    def __init__(self, cell=nn.LSTM, input_size=1, hidden_size=64):
        super().__init__()
        self.rnn = cell(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

def train_forecaster(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=1e-3, batch_size=64):
    model = model.to(device)
    Xt, yt = torch.tensor(X_tr, device=device), torch.tensor(y_tr, device=device)[:, None]
    Xv, yv = torch.tensor(X_va, device=device), torch.tensor(y_va, device=device)[:, None]
    opt, loss_fn = torch.optim.Adam(model.parameters(), lr=lr), nn.MSELoss()
    best_loss, best_state = float("inf"), None
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad(); loss_fn(model(Xt[idx]), yt[idx]).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv), yv).item()
        if vl < best_loss:
            best_loss, best_state = vl, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)

@torch.no_grad()
def forecast(model, X_):
    model.eval()
    return model(torch.tensor(X_, device=device)).cpu().numpy()[:, 0]

X, y = make_windows(z, 28)
is_train, is_val, is_test = split_masks(28)
set_seed(42)
lstm = Forecaster()
train_forecaster(lstm, X[is_train], y[is_train], X[is_val], y[is_val])
mae_lstm = mae_rides(forecast(lstm, X[is_val]), y[is_val])
print(f"LSTM (28 dagar) val MAE: {mae_lstm:,.0f}")

### Övning 1
Fönstren är `[0,1,2]`, `[1,2,3]` och `[2,3,4]` med målen 3, 4 och 5.

In [ ]:
expected_X = [[[0], [1], [2]], [[1], [2], [3]], [[2], [3], [4]]]
expected_y = [3, 4, 5]
Xw, yw = make_windows(np.arange(6, dtype="float32"), 3)
assert np.array_equal(Xw, np.array(expected_X, dtype="float32")) and np.array_equal(yw, np.array(expected_y, dtype="float32"))
print("Rätt!")

### Övning 2

In [ ]:
for w in [7, 28, 56]:
    Xw, yw = make_windows(z, w)
    tr, va, _ = split_masks(w)
    set_seed(42)
    m = Forecaster()
    train_forecaster(m, Xw[tr], yw[tr], Xw[va], yw[va])
    print(f"WINDOW={w:2d}: val MAE {mae_rides(forecast(m, Xw[va]), yw[va]):,.0f}")

**Tolkning:** med `WINDOW=7` ser modellen veckomönstret exakt *en* gång. Den kan inte jämföra flera veckor, och en enda helgdag i fönstret stör mycket. Längre fönster ger mer sammanhang. Efter en viss längd blir vinsten liten, medan träningen blir långsammare och det blir färre träningsexempel.

### Övning 3

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(X[~is_test]))           # slumpvis blandning av allt utom test
cut = int(0.8 * len(idx))
X_nt, y_nt = X[~is_test], y[~is_test]
set_seed(42)
m = Forecaster()
train_forecaster(m, X_nt[idx[:cut]], y_nt[idx[:cut]], X_nt[idx[cut:]], y_nt[idx[cut:]])
print(f"slumpvis uppdelning: val MAE {mae_rides(forecast(m, X_nt[idx[cut:]]), y_nt[idx[cut:]]):,.0f}"
      f"   test 2019 {mae_rides(forecast(m, X[is_test]), y[is_test]):,.0f}")
print(f"tidsbaserad:         val MAE {mae_lstm:,.0f}"
      f"   test 2019 {mae_rides(forecast(lstm, X[is_test]), y[is_test]):,.0f}")

**Vad vi ser:** här blir den slumpvisa val-siffran *inte* nämnvärt för optimistisk. Den ligger nära sin test-siffra, och modellen blir till och med något bättre på 2019. Varför?
- **Modellen kan inte fuska.** Varje prediktion bygger bara på de 28 föregående dagarna, och en liten LSTM med 17 000 parametrar kan inte lära sig enskilda dagar utantill.
- **Den har tränat på nyare data.** Den slumpvisa modellen har sett 2017–2018, som ligger närmast 2019. Det är en egen lärdom: **träna om på all data fram till idag innan modellen används.**

**Varför ändå tidsbaserad uppdelning?** Den slumpvisa valideringen mäter "fyll i en lucka mitt i historiken", inte "förutsäg framtiden". Att det råkade bli lika här är tur, inte en garanti. Läckaget blir allvarligt när:
- modellen kan lära sig utantill (stora modeller, få exempel),
- features innehåller information från framtiden (till exempel ett rullande medelvärde som *centreras* kring dagen, eller statistik beräknad på hela serien),
- datan ändrar karaktär över tid (trender och strukturella brott som pandemin 2020). En slumpvis val-mängd innehåller då samma regim som train, medan framtiden inte gör det.

Den tidsbaserade uppdelningen är den enda som speglar hur modellen faktiskt används.

### Övning 4

In [ ]:
dow = np.eye(7, dtype="float32")[series.index.day_of_week]          # (dagar, 7)
features = np.concatenate([z[:, None], dow], axis=1)                 # (dagar, 8)
X8, y8 = make_windows(features, 28)                                  # X8: (N, 28, 8), y8: (N, 8)
y8 = y8[:, 0]                                                        # målet är bara antalet resor
set_seed(42)
m = Forecaster(input_size=8)
train_forecaster(m, X8[is_train], y8[is_train], X8[is_val], y8[is_val])
mae_with_dow = mae_rides(forecast(m, X8[is_val]), y8[is_val])
assert mae_with_dow < mae_lstm, (mae_with_dow, mae_lstm)
print(f"Rätt! {mae_with_dow:,.0f} jämfört med {mae_lstm:,.0f}")

Veckodagen gör det lättare för modellen att placera sig i veckan. Nästa steg vore en **helgdagsflagga**, eftersom det var där de största felen fanns. Det kräver en lista över helgdagar, till exempel från paketet `holidays`.